Import Global setting

In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from itertools import product

RANDOM_STATE = 42

Load train/val/test features

In [17]:
train_feat = pd.read_parquet("data/train_features.parquet")
val_feat = pd.read_parquet("data/val_features.parquet")
full_train_feat = pd.read_parquet("data/full_train_features.parquet")
test_feat = pd.read_parquet("data/full_test_features.parquet")

Feature selection

In [18]:
# These columns are not used as model inputs.
# srch_id and prop_id are kept separately for grouping/submission.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Keep only numeric columns as model features.
feature_cols = [
    col for col in train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(train_feat[col])
]

# Make sure validation has exactly the same features.
missing_in_val = set(feature_cols) - set(val_feat.columns)
extra_in_val = set(val_feat.columns) - set(train_feat.columns)

print("Number of features:", len(feature_cols))
print("Missing in validation:", missing_in_val)
print("Extra in validation:", len(extra_in_val))

print(feature_cols[:50])

Number of features: 169
Missing in validation: set()
Extra in validation: 0
['site_id', 'visitor_location_country_id', 'visitor_hist_starrating', 'visitor_hist_adr_usd', 'prop_country_id', 'prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'price_usd', 'promotion_flag', 'srch_destination_id', 'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'srch_query_affinity_score', 'orig_destination_distance', 'random_bool', 'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff', 'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff', 'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff', 'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff', 'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff', 'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff', 'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff', 'comp8_rate', 'comp8_inv', 'comp8

Preparing Ranking Model Inputs

In [19]:
# LightGBM ranker needs rows sorted by search group to identify which rows belong to the same search.
train_feat = train_feat.sort_values("srch_id").reset_index(drop=True)
val_feat = val_feat.sort_values("srch_id").reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["relevance"].astype(int)

X_val = val_feat[feature_cols]
y_val = val_feat["relevance"].astype(int)

# Group sizes are needed for LightGBM ranker to know how many rows belong to each search group.
group_train = train_feat.groupby("srch_id").size().to_numpy()
group_val = val_feat.groupby("srch_id").size().to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of train groups:", len(group_train))
print("Number of validation groups:", len(group_val))
print("First 10 group sizes:", group_train[:10])

X_train: (3980039, 169)
X_val: (978308, 169)
Number of train groups: 159836
Number of validation groups: 39959
First 10 group sizes: [28 32 21 33 28 31 29 33 34 16]


Evaluate the ranking quality of the model using NDCG@k metric.

In [20]:
def dcg_at_k(relevances, k=5):
    """
    Computes DCG@k for one ranked list.
    """
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = (2 ** relevances - 1)
    return np.sum(gains / discounts)


def ndcg_at_k_for_group(y_true, y_score, k=5):
    """
    Computes NDCG@k for one search group.
    """
    order = np.argsort(-y_score)
    ranked_relevance = y_true[order]

    ideal_order = np.argsort(-y_true)
    ideal_relevance = y_true[ideal_order]

    dcg = dcg_at_k(ranked_relevance, k=k)
    idcg = dcg_at_k(ideal_relevance, k=k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_ndcg_at_k(df, y_true_col, y_score_col, group_col="srch_id", k=5):
    """
    Computes mean NDCG@k over all searches.
    """

    scores = []

    for _, group in df.groupby(group_col):
        y_true = group[y_true_col].to_numpy()
        y_score = group[y_score_col].to_numpy()

        scores.append(ndcg_at_k_for_group(y_true, y_score, k=k))

    return np.mean(scores)

Binary Classification model - LGBMClassifier Model

In [21]:
# ---------------------------------------------------
# Binary classification target
# ---------------------------------------------------

y_train_cls = (
    (train_feat["click_bool"] == 1) |
    (train_feat["booking_bool"] == 1)
).astype(int)

y_val_cls = (
    (val_feat["click_bool"] == 1) |
    (val_feat["booking_bool"] == 1)
).astype(int)

# ---------------------------------------------------
# Train classifier
# ---------------------------------------------------
param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.05, 0.03],
    "min_child_samples": [50, 100]
}

keys = list(param_grid.keys())

experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

classifier_results = []

for i, params in enumerate(experiments, start=1):
    print(f"Training classifier {i}/{len(experiments)}")
    print(params)

    classifier = lgb.LGBMClassifier(
        objective="binary",
        device="cpu",

        n_estimators=1000,

        subsample=0.8,
        colsample_bytree=0.8,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **params
    )

    classifier.fit(
        X_train,
        y_train_cls,
        eval_set=[(X_val, y_val_cls)],
        eval_metric="auc",

        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    )

    # ---------------------------------------------------
    # Predict probabilities
    # ---------------------------------------------------

    preds = classifier.predict_proba(
            X_val,
            num_iteration=classifier.best_iteration_
        )[:, 1]


    # ---------------------------------------------------
    # Evaluate ranking quality using NDCG
    # ---------------------------------------------------

    val_classifier_eval = val_feat[
            ["srch_id", "prop_id", "relevance"]
        ].copy()

    val_classifier_eval["prediction"] = preds

    classifier_ndcg = mean_ndcg_at_k(
        val_classifier_eval,
        y_true_col="relevance",
        y_score_col="prediction",
        group_col="srch_id",
        k=5
    )

    classifier_results.append({
            "experiment": i,

            "num_leaves": params["num_leaves"],
            "learning_rate": params["learning_rate"],
            "min_child_samples": params["min_child_samples"],

            "best_iteration": classifier.best_iteration_,
            "validation_ndcg@5": classifier_ndcg
        })

classifier_results = pd.DataFrame(classifier_results)

classifier_results = classifier_results.sort_values(
        "validation_ndcg@5",
        ascending=False
    )

best_classifier_parameters = {
    "num_leaves": int(classifier_results.iloc[0]["num_leaves"]),
    "learning_rate": float(classifier_results.iloc[0]["learning_rate"]),
    "min_child_samples": int(classifier_results.iloc[0]["min_child_samples"]),
    "n_estimators": int(classifier_results.iloc[0]["best_iteration"])
}

print(best_classifier_parameters)
display(classifier_results)

Training classifier 1/8
{'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 50}
[LightGBM] [Info] Number of positive: 177702, number of negative: 3802337
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.391331 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16588
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 168
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.044648 -> initscore=-3.063263
[LightGBM] [Info] Start training from score -3.063263
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.749254	valid_0's binary_logloss: 0.16489
Early stopping, best iteration is:
[110]	valid_0's auc: 0.749686	valid_0's binary_logloss: 0.164821
Training classifier 2/8
{'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100}
[LightGBM] [Info] Number of

,experiment,num_leaves,learning_rate,min_child_samples,best_iteration,validation_ndcg@5
7,8,63,0.03,100,166,0.373555
6,7,63,0.03,50,148,0.372937
4,5,63,0.05,50,93,0.372492
5,6,63,0.05,100,94,0.371863
1,2,31,0.05,100,125,0.371832
3,4,31,0.03,100,203,0.371473
2,3,31,0.03,50,190,0.370823
0,1,31,0.05,50,110,0.370019


LGBMRanker Model

In [54]:
# # ---------------------------------------------------
# # Hyperparameter grid
# # ---------------------------------------------------

# # param_grid = {
# #     "num_leaves": [31, 63, 127],
# #     "learning_rate": [0.05, 0.01, 0.1],
# #     "min_child_samples": [50, 100, 200]
# # }
# # -> num_leaves = 127, lr = 0.01, min_child_samples = 200 -> 0.382248


# # param_grid = {
# #     "num_leaves": [95, 127, 191, 255],
# #     "learning_rate": [0.005, 0.01, 0.02, 0.03],
# #     "min_child_samples": [150, 200, 300, 500]
# # }
# # -> num_leaves = 255, lr = 0.02, min_child_samples = 500 submitted -> 0.385801

# # param_grid = {
# #     "num_leaves": [255, 319],
# #     "learning_rate": [0.02, 0.025],
# #     "min_child_samples": [500, 700]
# # }
# # -> num_leaves = 255, lr = 0.02, min_child_samples = 500 -> 0.385801


# # param_grid = {
# #     "num_leaves": [191, 255, 319, 383],
# #     "learning_rate": [0.015, 0.02, 0.025, 0.03],
# #     "min_child_samples": [300, 500, 700, 900]
# # }
# # -> num_leaves = 383, lr = 0.02, min_child_samples = 900 -> 0.386670


# # Create all parameter combinations
# keys = list(param_grid.keys())

# experiments = [
#     dict(zip(keys, values))
#     for values in product(*param_grid.values())
# ]

# print("Total experiments:", len(experiments))

# # ---------------------------------------------------
# # Run experiments
# # ---------------------------------------------------

# tuning_results = []

# for i, params in enumerate(experiments, start=1):

#     print("=" * 60)
#     print(f"Experiment {i}/{len(experiments)}")
#     print(params)

#     ranker = lgb.LGBMRanker(
#         objective="lambdarank",
#         metric="ndcg",
#         ndcg_eval_at=[5],
#         boosting_type="gbdt",
#         device="cpu",
#         n_estimators=1000,

#         subsample=0.8,
#         colsample_bytree=0.8,

#         random_state=RANDOM_STATE,
#         n_jobs=-1,

#         **params
#     )

#     ranker.fit(
#         X_train,
#         y_train,
#         group=group_train,

#         eval_set=[(X_val, y_val)],
#         eval_group=[group_val],
#         eval_at=[5],

#         callbacks=[
#             lgb.early_stopping(stopping_rounds=50),
#             lgb.log_evaluation(period=100)
#         ]
#     )

#     # -------------------------
#     # Predict validation scores
#     # -------------------------

#     preds = ranker.predict(
#         X_val,
#         num_iteration=ranker.best_iteration_
#     )

#     # -------------------------
#     # Compute validation NDCG@5
#     # -------------------------

#     tmp = val_feat[["srch_id", "prop_id", "relevance"]].copy()

#     tmp["prediction"] = preds

#     score = mean_ndcg_at_k(
#         tmp,
#         y_true_col="relevance",
#         y_score_col="prediction",
#         group_col="srch_id",
#         k=5
#     )

#     # -------------------------
#     # Save results
#     # -------------------------

#     tuning_results.append({
#         "experiment": i,

#         "num_leaves": params["num_leaves"],
#         "learning_rate": params["learning_rate"],
#         "min_child_samples": params["min_child_samples"],

#         "best_iteration": ranker.best_iteration_,
#         "validation_ndcg@5": score
#     })

# # ---------------------------------------------------
# # Final results table
# # ---------------------------------------------------

# tuning_results = pd.DataFrame(tuning_results)

# tuning_results = tuning_results.sort_values(
#     "validation_ndcg@5",
#     ascending=False
# )

# best_parameters = {
#     "num_leaves": int(tuning_results.iloc[0]["num_leaves"]),
#     "learning_rate": float(tuning_results.iloc[0]["learning_rate"]),
#     "min_child_samples": int(tuning_results.iloc[0]["min_child_samples"]),
#     "n_estimators": int(tuning_results.iloc[0]["best_iteration"])
# }

# display(tuning_results)

# ---------------------------------------------------
# Regularization tuning around the best model so far
# ---------------------------------------------------

# Best core parameters found in the previous grid search:
# num_leaves = 383, learning_rate = 0.02, min_child_samples = 900
# validation NDCG@5 = 0.386670

# ---------------------------------------------------
# Cell A: Regularization and sampling tuning
# ---------------------------------------------------
# We already found a strong core LightGBM Ranker setting:
# num_leaves = 383
# learning_rate = 0.02
# min_child_samples = 900
# validation NDCG@5 = 0.386670
#
# Now we keep these core parameters fixed and tune:
# - reg_lambda: L2 regularization
# - reg_alpha: L1 regularization
# - subsample: row sampling per tree
# - colsample_bytree: feature sampling per tree

from itertools import product
import pandas as pd
import lightgbm as lgb

# Best core parameters from previous grid search
base_params = {
    "num_leaves": 383,
    "learning_rate": 0.02,
    "min_child_samples": 900
}

# 4 x 2 x 2 x 2 = 32 experiments
param_grid = {
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
    "reg_alpha": [0.0, 0.1],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9]
}

# Create all parameter combinations
keys = list(param_grid.keys())

experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

print("Total experiments:", len(experiments))

# ---------------------------------------------------
# Run experiments
# ---------------------------------------------------

tuning_results = []

for i, params in enumerate(experiments, start=1):

    # Combine fixed best parameters with the current regularization/sampling parameters
    all_params = {**base_params, **params}

    print("=" * 60)
    print(f"Experiment {i}/{len(experiments)}")
    print(all_params)

    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",
        device="cpu",

        # 1000 is enough here because the previous best stopped at 202 trees
        n_estimators=1000,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **all_params
    )

    ranker.fit(
        X_train,
        y_train,
        group=group_train,

        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[5],

        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    # ---------------------------------------------------
    # Predict validation scores
    # ---------------------------------------------------

    preds = ranker.predict(
        X_val,
        num_iteration=ranker.best_iteration_
    )

    # ---------------------------------------------------
    # Compute validation NDCG@5
    # ---------------------------------------------------

    tmp = val_feat[["srch_id", "prop_id", "relevance"]].copy()
    tmp["prediction"] = preds

    score = mean_ndcg_at_k(
        tmp,
        y_true_col="relevance",
        y_score_col="prediction",
        group_col="srch_id",
        k=5
    )

    # ---------------------------------------------------
    # Save experiment results
    # ---------------------------------------------------

    tuning_results.append({
        "experiment": i,

        "num_leaves": all_params["num_leaves"],
        "learning_rate": all_params["learning_rate"],
        "min_child_samples": all_params["min_child_samples"],

        "reg_lambda": all_params["reg_lambda"],
        "reg_alpha": all_params["reg_alpha"],
        "subsample": all_params["subsample"],
        "colsample_bytree": all_params["colsample_bytree"],

        "best_iteration": ranker.best_iteration_,
        "validation_ndcg@5": score
    })

# ---------------------------------------------------
# Final results table
# ---------------------------------------------------

tuning_results = pd.DataFrame(tuning_results)

tuning_results = tuning_results.sort_values(
    "validation_ndcg@5",
    ascending=False
).reset_index(drop=True)

best_row = tuning_results.iloc[0]

best_parameters = {
    "num_leaves": int(best_row["num_leaves"]),
    "learning_rate": float(best_row["learning_rate"]),
    "min_child_samples": int(best_row["min_child_samples"]),

    "reg_lambda": float(best_row["reg_lambda"]),
    "reg_alpha": float(best_row["reg_alpha"]),
    "subsample": float(best_row["subsample"]),
    "colsample_bytree": float(best_row["colsample_bytree"]),

    "n_estimators": int(best_row["best_iteration"])
}

print("Best parameters:")
print(best_parameters)

print("\nBest validation NDCG@5:")
print(best_row["validation_ndcg@5"])

display(tuning_results)

Total experiments: 32
Experiment 1/32
{'num_leaves': 383, 'learning_rate': 0.02, 'min_child_samples': 900, 'reg_lambda': 0.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.519241 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16588
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 168
Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.38379
[200]	valid_0's ndcg@5: 0.386424
Early stopping, best iteration is:
[202]	valid_0's ndcg@5: 0.38667


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Experiment 2/32
{'num_leaves': 383, 'learning_rate': 0.02, 'min_child_samples': 900, 'reg_lambda': 0.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.9}


...

/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.486836 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16588
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 168
Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.381005
[200]	valid_0's ndcg@5: 0.38069
Early stopping, best iteration is:
[153]	valid_0's ndcg@5: 0.382003


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Best parameters:
{'num_leaves': 383, 'learning_rate': 0.02, 'min_child_samples': 900, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'subsample': 0.9, 'colsample_bytree': 0.8, 'n_estimators': 286}

Best validation NDCG@5:
0.3871559317282132


,experiment,num_leaves,learning_rate,min_child_samples,reg_lambda,reg_alpha,subsample,colsample_bytree,best_iteration,validation_ndcg@5
0,27,383,0.02,900,10.0,0.0,0.9,0.8,286,0.387156
1,25,383,0.02,900,10.0,0.0,0.8,0.8,286,0.387156
2,17,383,0.02,900,5.0,0.0,0.8,0.8,239,0.387124
3,19,383,0.02,900,5.0,0.0,0.9,0.8,239,0.387124
4,1,383,0.02,900,0.0,0.0,0.8,0.8,202,0.386670
5,3,383,0.02,900,0.0,0.0,0.9,0.8,202,0.386670
6,9,383,0.02,900,1.0,0.0,0.8,0.8,243,0.386425
7,11,383,0.02,900,1.0,0.0,0.9,0.8,243,0.386425
8,28,383,0.02,900,10.0,0.0,0.9,0.9,244,0.386328
9,26,383,0.02,900,10.0,0.0,0.8,0.9,244,0.386328


In [55]:
# # Rebuild feature list from full training data.
# final_feature_cols = [
#     col for col in full_train_feat.columns
#     if col not in NON_FEATURE_COLS
#     and pd.api.types.is_numeric_dtype(full_train_feat[col])
# ]

# # Ensure test has all final feature columns.
# missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
# print("Missing features in test:", missing_in_test)

# # Sort by srch_id to ensure correct grouping for LightGBM ranker.
# full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
# test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

# X_full = full_train_feat[final_feature_cols]
# y_full = full_train_feat["relevance"].astype(int)
# group_full = full_train_feat.groupby("srch_id").size().to_numpy()

# X_test = test_feat[final_feature_cols]

# print("X_full:", X_full.shape)
# print("X_test:", X_test.shape)
# print("Number of final features:", len(final_feature_cols))

# ---------------------------------------------------
# Prepare full training and test matrices for final model
# ---------------------------------------------------
# This cell prepares the full training set and test set using the same feature columns.
# It does not train the model yet.

# Columns that should not be used as model input.
# srch_id and prop_id are identifiers.
# date_time is not used directly because we already extracted date/time features.
# click_bool, booking_bool, gross_bookings_usd, position, and relevance are target-related or unavailable in test.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Rebuild feature list from full training data.
# Only numeric columns are used by the final LightGBM Ranker.
final_feature_cols = [
    col for col in full_train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(full_train_feat[col])
]

# Ensure test has all final feature columns.
missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
print("Missing features in test:", missing_in_test)

if len(missing_in_test) > 0:
    raise ValueError("Some final training features are missing in the test set.")

# Sort by srch_id to ensure correct grouping for LightGBM ranker.
# LightGBM needs group sizes to match the row order.
full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

# Create final training matrix, target, group sizes, and test matrix.
X_full = full_train_feat[final_feature_cols]
y_full = full_train_feat["relevance"].astype(int)
group_full = full_train_feat.groupby("srch_id").size().to_numpy()

X_test = test_feat[final_feature_cols]

print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("Number of final features:", len(final_feature_cols))

Missing features in test: []
X_full: (4958347, 169)
X_test: (4959183, 169)
Number of final features: 169


In [56]:
# # Use the best iteration from validation if available.
# # If early stopping stopped at e.g. 350 trees, train final model with that many trees.
# best_cls_n_estimators = ranker.best_iteration_

# final_classifier = lgb.LGBMClassifier(
#     objective="binary",

#     n_estimators=best_cls_n_estimators,
#     num_leaves=best_classifier_parameters["num_leaves"],
#     learning_rate=best_classifier_parameters["learning_rate"],
#     min_child_samples=best_classifier_parameters["min_child_samples"],

#     subsample=0.8,
#     colsample_bytree=0.8,

#     random_state=RANDOM_STATE,
#     n_jobs=-1
# )

In [57]:
# # Use the best iteration from validation if available.
# # If early stopping stopped at e.g. 350 trees, train final model with that many trees.
# # best_n_estimators = ranker.best_iteration_
# best_n_estimators = best_parameters["n_estimators"] #------------------------------------------------------------------!!!!!!!!!!

# print("Training final model with n_estimators =", best_n_estimators)

# final_ranker = lgb.LGBMRanker(
#     objective="lambdarank",
#     metric="ndcg",
#     ndcg_eval_at=[5],
#     boosting_type="gbdt",

#     n_estimators=best_n_estimators,
#     learning_rate=best_parameters["learning_rate"],
#     num_leaves=best_parameters["num_leaves"],
#     max_depth=-1,
#     min_child_samples=best_parameters["min_child_samples"],
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=RANDOM_STATE,
#     n_jobs=-1
# )

# final_ranker.fit(
#     X_full,
#     y_full,
#     group=group_full
# )

# ---------------------------------------------------
# Train final LightGBM Ranker on the full training set
# ---------------------------------------------------
# This uses the best parameters found during validation tuning.
# The final model is trained on the full training data.

best_n_estimators = best_parameters["n_estimators"]

print("Training final model with parameters:")
print(best_parameters)

final_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[5],
    boosting_type="gbdt",

    n_estimators=best_n_estimators,
    learning_rate=best_parameters["learning_rate"],
    num_leaves=best_parameters["num_leaves"],
    max_depth=-1,
    min_child_samples=best_parameters["min_child_samples"],

    reg_lambda=best_parameters["reg_lambda"],
    reg_alpha=best_parameters["reg_alpha"],
    subsample=best_parameters["subsample"],
    colsample_bytree=best_parameters["colsample_bytree"],

    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_ranker.fit(
    X_full,
    y_full,
    group=group_full
)

print("Final ranker trained.")

Training final model with parameters:
{'num_leaves': 383, 'learning_rate': 0.02, 'min_child_samples': 900, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'subsample': 0.9, 'colsample_bytree': 0.8, 'n_estimators': 286}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.574145 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 16596
[LightGBM] [Info] Number of data points in the train set: 4958347, number of used features: 167
Final ranker trained.


In [58]:
# Check which datetime columns are still in test_feat
test_feat.select_dtypes(include=["datetime64", "datetime64[ns]"]).columns

Index(['date_time'], dtype='str')

In [59]:
# test_scores = final_ranker.predict(test_feat)

feature_cols = final_ranker.feature_name_

test_scores = final_ranker.predict(test_feat[feature_cols])

submission = test_feat[["srch_id", "prop_id"]].copy()
submission["score"] = test_scores

submission = submission.sort_values(
    ["srch_id", "score"],
    ascending=[True, False]
)

# submission = submission[["srch_id", "prop_id"]]
# submission.to_csv("submission.csv", index=False)

# submission = test_feat[["srch_id", "prop_id"]].copy()
# submission["score"] = test_scores

# # Sort hotels within each search by predicted score descending.
# submission = submission.sort_values(
#     ["srch_id", "score"],
#     ascending=[True, False]
# )

# # Required Kaggle format:
# # SearchId,PropertyId
# submission = submission.rename(columns={
#     "srch_id": "SearchId",
#     "prop_id": "PropertyId"
# })

submission = submission[["srch_id", "prop_id"]]

display(submission.head(30))

submission_path = "submission_lgbm_ranker.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)

/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


,srch_id,prop_id
6,1,99484
16,1,54937
27,1,28181
26,1,61934
20,1,24194
21,1,34263
23,1,50162
7,1,95031
2,1,90385
22,1,5543


Saved submission to: submission_lgbm_ranker.csv
Submission shape: (4959183, 2)


In [60]:
final_feature_importance = pd.DataFrame({
    # "feature": final_feature_cols,
    "feature": final_ranker.feature_name_,
    "importance": final_ranker.feature_importances_
}).sort_values("importance", ascending=False)

display(final_feature_importance.head(50))

final_feature_importance.to_csv("feature_importance_lgbm_ranker.csv", index=False)
print("Saved feature importance to feature_importance_lgbm_ranker.csv") 

,feature,importance
10,prop_log_historical_price,3661
154,prop_id_median_price,3306
160,prop_id_mean_location_score2,3132
156,prop_id_mean_log_price,3110
11,price_usd,2949
152,prop_id_count,2878
147,location2_diff_from_search_mean,2733
129,price_pct_rank_in_search,2670
153,prop_id_mean_price,2561
137,star_pct_rank_in_search,2530


Saved feature importance to feature_importance_lgbm_ranker.csv
